In [3]:
!pip3 install "urllib3<2"

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 144 kB 857 kB/s eta 0:00:01
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.6.3
    Uninstalling urllib3-2.6.3:
      Successfully uninstalled urllib3-2.6.3
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import networkx as nx
import yfinance as yf
import warnings

from datetime import timedelta
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

In [8]:
print("--- STEP 1: LOADING AND CLEANING DATASET ---")

try:
    df = pd.read_csv('senate_stock_discosures.csv')
    print(f"Dataset loaded successfully. Total records: {len(df)}")
except FileNotFoundError:
    print("Warning: CSV not found. Using synthetic data for pipeline validation.")
    # Synthetic records cover a range of senators and dates across multiple years
    # so that yfinance can fetch real historical prices for testing the pipeline.
    df = pd.DataFrame({
        'first_name': [
            'John', 'Ron', 'Tommy', 'Mitch', 'Shelley', 'William',
            'Dianne', 'Richard', 'Kelly', 'Mark', 'John', 'Pat',
            'James', 'Susan', 'David'
        ],
        'last_name': [
            'Hickenlooper', 'Wyden', 'Tuberville', 'McConnell', 'Capito', 'Cassidy',
            'Feinstein', 'Burr', 'Loeffler', 'Warner', 'Barrasso', 'Roberts',
            'Inhofe', 'Collins', 'Perdue'
        ],
        'ticker': [
            'PYPL', 'MSFT', 'QCOM', 'AAPL', 'NVDA', 'JNJ',
            'GOOGL', 'MRNA', 'XOM', 'JPM', 'CVX', 'LMT',
            'BA', 'PFE', 'AAPL'
        ],
        'transaction': [
            'Sale', 'Sale', 'Purchase', 'Purchase', 'Sale', 'Purchase',
            'Sale', 'Sale', 'Sale', 'Sale', 'Purchase', 'Purchase',
            'Purchase', 'Sale', 'Sale'
        ],
        'asset_value_high': [
            250000, 500000, 100000, 150000, 300000, 200000,
            400000, 1000000, 500000, 300000, 150000, 250000,
            200000, 100000, 350000
        ],
        'asset_value_low': [
            100001, 250001, 50001, 100001, 150001, 100001,
            200001, 500001, 250001, 150001, 100001, 100001,
            100001, 50001, 200001
        ],
        'owner': [
            'Self', 'Spouse', 'Joint', 'Spouse', 'Self', 'Joint',
            'Self', 'Self', 'Self', 'Spouse', 'Joint', 'Self',
            'Self', 'Spouse', 'Self'
        ],
        'transaction_date': [
            '2020-02-01', '2020-02-03', '2020-03-15', '2021-02-20',
            '2020-01-20', '2021-11-05', '2022-06-10', '2020-01-24',
            '2020-01-28', '2020-02-05', '2021-07-12', '2021-09-03',
            '2022-01-15', '2021-04-20', '2020-08-10'
        ]
    })

# Parse dates and derive temporal features used throughout the pipeline
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['year']  = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month

# Standardize monetary columns; fill missing values with zero before computing
# the mid-point estimate which serves as our proxy for transaction size
df['asset_value_high'] = pd.to_numeric(df['asset_value_high'], errors='coerce').fillna(0)
df['asset_value_low']  = pd.to_numeric(df['asset_value_low'],  errors='coerce').fillna(0)
df['estimated_volume'] = (df['asset_value_high'] + df['asset_value_low']) / 2

print(f"\nDate range  : {df['transaction_date'].min().date()} → {df['transaction_date'].max().date()}")
print(f"Senators    : {df[['first_name','last_name']].drop_duplicates().shape[0]}")
print(f"Unique assets: {df['ticker'].nunique()}")
print(f"Total trades : {len(df)}")


--- STEP 1: LOADING AND CLEANING DATASET ---
Dataset loaded successfully. Total records: 4699

Date range  : 2012-09-13 → 2024-04-30
Senators    : 40
Unique assets: 793
Total trades : 4699


In [11]:

# The central empirical question in insider trading research is whether a
# senator's trades generate returns that exceed what a random investor would
# earn from broad market exposure. We operationalize this as "alpha":
#
#   alpha_i = R_stock_i(t, t+30) - R_SPY(t, t+30)
#
# where t is the transaction date and the window is 30 calendar days.
# A significant positive alpha on purchases (or negative on sales before drops)
# suggests that the trading decision incorporated information not yet reflected
# in publicly available prices -- the core definition of informed trading.
#
# This approach mirrors the abnormal return methodology in the seminal
# event study literature (Fama, Fisher, Jensen & Roll, 1969) and is the
# same metric used by Ziobrowski et al. (2004) and Eggers & Hainmueller (2014)
# in their empirical studies of congressional trading performance.
# =============================================================================
import logging

# 1. BÜTÜN KIRMIZI UYARILARI VE 404 HATALARINI SUSTUR
logging.getLogger('yfinance').setLevel(logging.CRITICAL)

print("--- STEP 2: COMPUTING MARKET-ADJUSTED RETURNS (ALPHA) ---")
print("Downloading historical price data. This may take a minute...\n")

ALPHA_WINDOW_DAYS = 30   # forward window in calendar days
BENCHMARK_TICKER  = 'SPY'  # S&P 500 as the market proxy

# Determine the full date range we need to cover
date_min = df['transaction_date'].min() - timedelta(days=5)
date_max = df['transaction_date'].max() + timedelta(days=ALPHA_WINDOW_DAYS + 15)

# Download the benchmark once for the entire period to avoid repeated API calls
benchmark_raw = yf.download(BENCHMARK_TICKER, start=date_min, end=date_max, progress=False)
benchmark_raw.index = benchmark_raw.index.tz_localize(None).normalize()

# Cache stock price histories by ticker -- this is much more efficient than
# downloading a fresh price series for every individual transaction row
unique_tickers = df['ticker'].dropna().astype(str).str.strip().unique()
price_cache = {}

for ticker in unique_tickers:
    try:
        data = yf.download(ticker, start=date_min, end=date_max, progress=False)
        if not data.empty:
            data.index = data.index.tz_localize(None).normalize()
            price_cache[ticker] = data
    except Exception:
        pass  # silently skip delisted or invalid tickers

print(f"Price data cached: {len(price_cache)} / {len(unique_tickers)} tickers.\n")


def compute_alpha(row, price_cache, benchmark, window=30):
    """
    Computes the 30-day forward market-adjusted return.
    Includes fail-safes for newer yfinance MultiIndex/Series formats.
    """
    ticker = str(row['ticker']).strip()
    trade_date = pd.to_datetime(row['transaction_date'])

    if ticker not in price_cache or benchmark.empty:
        return np.nan

    stock = price_cache[ticker]
    
    # Entry: first trading day on or after the transaction date
    stock_entry_data = stock[stock.index >= trade_date]
    bench_entry_data = benchmark[benchmark.index >= trade_date]
    
    if stock_entry_data.empty or bench_entry_data.empty:
        return np.nan

    # Yfinance versiyon farklılıkları için güvenli veri çekme fonksiyonu
    def get_scalar(val):
        if isinstance(val, pd.Series):
            return float(val.iloc[0])
        elif isinstance(val, pd.DataFrame):
            return float(val.iloc[0, 0])
        return float(val)

    p_stock_entry = get_scalar(stock_entry_data['Close'].iloc[0])
    p_bench_entry = get_scalar(bench_entry_data['Close'].iloc[0])

    # Exit: first trading day on or after (transaction_date + window)
    exit_date = trade_date + timedelta(days=window)
    stock_exit_data = stock[stock.index >= exit_date]
    bench_exit_data = benchmark[benchmark.index >= exit_date]
    
    if stock_exit_data.empty or bench_exit_data.empty:
        return np.nan

    p_stock_exit = get_scalar(stock_exit_data['Close'].iloc[0])
    p_bench_exit = get_scalar(bench_exit_data['Close'].iloc[0])

    r_stock = (p_stock_exit / p_stock_entry - 1.0) * 100.0
    r_bench = (p_bench_exit / p_bench_entry - 1.0) * 100.0

    return r_stock - r_bench


df['alpha'] = df.apply(
    compute_alpha, axis=1,
    price_cache=price_cache,
    benchmark=benchmark_raw,
    window=ALPHA_WINDOW_DAYS
)

coverage = df['alpha'].notna().sum()
print(f"Alpha coverage : {coverage} / {len(df)} transactions ({100*coverage/len(df):.1f}%)")
print(f"Mean alpha     : {df['alpha'].mean():.3f}%")
print(f"Std alpha      : {df['alpha'].std():.3f}%")
print(f"Min / Max      : {df['alpha'].min():.2f}% / {df['alpha'].max():.2f}%")

--- STEP 2: COMPUTING MARKET-ADJUSTED RETURNS (ALPHA) ---

Price data cached: 733 / 793 tickers.

Alpha coverage : 4476 / 4699 transactions (95.3%)
Mean alpha     : 0.172%
Std alpha      : 9.424%
Min / Max      : -52.86% / 233.01%


In [12]:
print("\n--- STEP 3: FEATURE ENGINEERING ---")

# --- Proxy / account-type flag ---
# Trades executed through a spouse, child, or joint account may indicate
# an attempt to distance the senator from the transaction while still
# benefiting from non-public information. We encode this as a binary flag.
df['owner_clean'] = df['owner'].fillna('Self').astype(str)
df['is_proxy'] = df['owner_clean'].str.contains(
    'Spouse|Joint|Child', case=False, na=False
).astype(int)

# --- Trade direction ---
df['is_buy_order'] = df['transaction'].astype(str).str.lower().apply(
    lambda x: 1 if ('purchase' in x or 'buy' in x) else 0
)

# --- Options indicator ---
# Options trades (calls, puts) require the most precise timing to be profitable,
# making them a particularly sensitive signal for informed trading.
df['is_option'] = df['transaction'].astype(str).str.lower().str.contains(
    'option|call|put', na=False
).astype(int)

# --- Log-transformed volume ---
# Transaction size follows a heavily right-skewed distribution.
# We apply a log transformation to bring extreme values into a range
# that machine learning algorithms can process without numerical distortion.
df['log_volume'] = np.log1p(df['estimated_volume'])

# --- Sector classification ---
# Senators sit on committees that oversee specific industries.
# Sector membership captures whether a trade is in a regulated industry
# where the senator may hold an informational advantage.
SECTOR_MAP = {
    'AAPL': 'Technology', 'MSFT': 'Technology', 'NVDA': 'Technology',
    'INTC': 'Technology', 'QCOM': 'Technology', 'PYPL': 'Technology',
    'GOOGL': 'Technology', 'AMZN': 'Technology', 'META': 'Technology',
    'TSLA': 'Technology', 'AMD': 'Technology', 'CRM': 'Technology',
    'JNJ': 'Healthcare',  'MRNA': 'Healthcare', 'PFE': 'Healthcare',
    'ABBV': 'Healthcare', 'UNH': 'Healthcare', 'BMY': 'Healthcare',
    'JPM': 'Finance',     'BAC': 'Finance',     'GS':  'Finance',
    'WFC': 'Finance',     'MS':  'Finance',     'C':   'Finance',
    'XOM': 'Energy',      'CVX': 'Energy',      'COP': 'Energy',
    'BA':  'Defense',     'LMT': 'Defense',     'RTX': 'Defense',
    'NOC': 'Defense',     'GD':  'Defense',
}
df['sector'] = df['ticker'].map(SECTOR_MAP).fillna('Other')
sector_encoder = LabelEncoder()
df['sector_code'] = sector_encoder.fit_transform(df['sector'])

print("Feature engineering complete. New columns added:")
print("  is_proxy, is_buy_order, is_option, log_volume, sector, sector_code")



--- STEP 3: FEATURE ENGINEERING ---
Feature engineering complete. New columns added:
  is_proxy, is_buy_order, is_option, log_volume, sector, sector_code


In [13]:
# =============================================================================
# We define a transaction as "statistically suspicious" if it generated an
# abnormal market-adjusted return exceeding a ±5% threshold in the direction
# consistent with informed trading:
#
#   Purchase + alpha > +5%  → bought just before a significant outperformance
#   Sale     + alpha < -5%  → sold just before a significant underperformance
#
# The 5% threshold is deliberately conservative -- it targets only the most
# extreme observations while controlling for false positives.
# Note: the label is constructed from forward-looking alpha (post-trade return),
# but the model is trained exclusively on pre-trade features to prevent
# data leakage. Alpha itself is never included as a training feature.
# =============================================================================

print("\n--- STEP 4: CONSTRUCTING THE TARGET VARIABLE ---")

ALPHA_THRESHOLD = 5.0  # percentage points above/below market

def label_suspicious(row):
    """
    Returns 1 if the transaction outcome is consistent with informed trading,
    0 otherwise. Only trades with a non-null alpha value are labeled positive.
    """
    alpha = row.get('alpha', np.nan)
    trade = str(row.get('transaction', '')).lower()

    if pd.isna(alpha):
        return 0
    if ('sale' in trade or 'sell' in trade) and alpha < -ALPHA_THRESHOLD:
        return 1
    if ('purchase' in trade or 'buy' in trade) and alpha > ALPHA_THRESHOLD:
        return 1
    return 0


df['is_suspicious'] = df.apply(label_suspicious, axis=1)

n_suspicious = df['is_suspicious'].sum()
n_total      = len(df)
print(f"Suspicious transactions : {n_suspicious} / {n_total} ({100*n_suspicious/n_total:.1f}%)")
print(f"Normal transactions     : {n_total - n_suspicious} / {n_total}")
print("\nNote: class imbalance is expected and will be handled via class_weight='balanced'.")



--- STEP 4: CONSTRUCTING THE TARGET VARIABLE ---
Suspicious transactions : 947 / 4699 (20.2%)
Normal transactions     : 3752 / 4699

Note: class imbalance is expected and will be handled via class_weight='balanced'.


In [14]:
df.to_csv('senate_processed_data_with_alpha.csv', index=False)
print("data is captured succesfully")

data is captured succesfully
